##### We have a Knowledge Base for a company shared drive. Task is to build an AI Knowledge Worker
##### We have to:-
    - Read names of products and employees
    - if qquestions refer to employee or profucst bu name
    - add relevant details to the prompt

In [27]:
from openai import OpenAI
import gradio as gr 

In [28]:
base_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama",base_url=base_url)

In [39]:
import glob
from pathlib import Path
knowledge = {}

filenames = glob.glob("knowledge-base/employees/*")

for filename in filenames:
    name = Path(filename).stem.split(" ")[-1].lower()
    with open(filename, "r", encoding="UTF-8") as f:
        knowledge[name] = f.read()



In [40]:
print(filenames[0],"\n")
print(Path(filenames[0]),"\n")
print(Path(filenames[0]).stem,"\n") # stem extracts the filename without the extension
print(Path(filenames[0]).stem.split(" "),"\n")
print(Path(filenames[0]).stem.split(" ")[-1],"\n")

knowledge-base/employees\Alex Chen.md 

knowledge-base\employees\Alex Chen.md 

Alex Chen 

['Alex', 'Chen'] 

Chen 



In [45]:
## Do the same for products
## Create a System Prefix for the InsureTech Company 

filenames = glob.glob("knowledge-base/products/*")

for filename in filenames:
    name = Path(filename).stem.split(" ")[-1].lower()
    with open(filename,"r",encoding="UTF-8") as f:
        knowledge[name] = f.read()

In [46]:
SYSTEM_PREFIx = """
You are an assistant for an Insurance Tech Company called InsureTech.
You are expert at answering question about the employees and products of the company.
You have access the company's knowledge base which conatins information about the employees and products.
You are provided with additional context that might help you answer the user's question.
Give brief accurate answers to the user question if you dont know the answer then say so.

Relevant Context:
"""

In [49]:
def get_relevant_context_simple(query):
    text = ''.join(ch for ch in query if ch.isalnum() or ch.isspace()) ## Sharper ways to do this is Regular Expressions
    words = text.lower().split()
    # print(words)
    relevant_context = []
    for word in words:
        # print(word)
        if word in knowledge:
            # print(word)
            relevant_context.append(knowledge[word])

    return relevant_context
    

In [50]:
get_relevant_context_simple("Who is Lancaster and what is carllm?")

["# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  \n\n- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market 

In [51]:
def additional_context(query):
    relevantContext = get_relevant_context_simple(query)
    if relevantContext:
        result = "The following information might be relevant to answer the question:\n\n"
        result += "\n\n".join(relevantContext)
    else:
        result = "No relevant information found in the knowledge base."
    return result

In [ ]:
def chat(message, history):
    systemPrompt = SYSTEM_PREFIx + additional_context(message)
    messages = [{"role":"system","content":systemPrompt}]+history+[{"role":"user","content": message}]
    response = ollama.chat.completions.create(
        model = "qwen3:30b",
        messages = messages
    )
    return response.choices[0].message.content

In [58]:
view = gr.ChatInterface(fn=chat).launch(inbrowser=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
